# 02 - Data Preprocessing

# Imports

In [60]:
#standard
import pandas as pd
import numpy as np

#Sentence Transformers for Classification / Regression Analysis
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
#Load Embedding Model
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 21346.64it/s]


In [46]:
#Homebrew Packages
#uncomment the below once we move into src
#from src.data_loader import load_data

from IPython import get_ipython

get_ipython().run_line_magic(
    "run",
    "01_Data_Loading.ipynb"
)


File selected successfully.
/Users/noa/Downloads/Drying Data Example Sheet - Sheet1.csv
/Users/noa/Downloads/Drying Data Example Sheet - Sheet1.csv
Detected delimiter: ','

Dataset Loaded Successfully
----------------------------------------
File type: .csv
Rows: 11
Columns: 8

Column Names:
['Unnamed: 0', 'Index', 'Material', 'Cake Volume (cubic inches)', 'Surface Area cubic inches', 'Nitrogen Flow (L/min)', 'Vacuum (mbar)', 'Drying Time (minutes)']


# Column Detection

In [47]:

def detect_column_types(df):

    """
    Detect numeric and categorical columns.
    """

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_columns = df.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    print("\nNumeric Columns:")
    print("-" * 40)
    print(numeric_columns)
    print("-" * 40)

    print("\n\nCategorical Columns:")
    print("-" * 40)
    print(categorical_columns)
    print("-" * 40)

    return {
        "numeric": numeric_columns,
        "categorical": categorical_columns
    }

In [48]:
detect_column_types(df)


Numeric Columns:
----------------------------------------
['Unnamed: 0', 'Index', 'Cake Volume (cubic inches)', 'Surface Area cubic inches', 'Nitrogen Flow (L/min)', 'Vacuum (mbar)', 'Drying Time (minutes)']
----------------------------------------


Categorical Columns:
----------------------------------------
['Material']
----------------------------------------


{'numeric': ['Unnamed: 0',
  'Index',
  'Cake Volume (cubic inches)',
  'Surface Area cubic inches',
  'Nitrogen Flow (L/min)',
  'Vacuum (mbar)',
  'Drying Time (minutes)'],
 'categorical': ['Material']}

In [49]:
df.head()

,Unnamed: 0,Index,Material,Cake Volume (cubic inches),Surface Area cubic inches,Nitrogen Flow (L/min),Vacuum (mbar),Drying Time (minutes)
0,NaN,1,GDC-1234,75,7.5,60,100,240
1,NaN,2,Naproxen,100,10.0,70,120,260
2,NaN,3,Acetopminiphen,125,12.5,60,250,300
3,NaN,4,GDC-1234,300,30.0,50,240,600
4,NaN,5,GDC-9876,125,12.5,45,100,500


# Detect Index Column Types

In [50]:
def detect_index_like_columns(df):

    """
    Detect columns that may behave like
    indices or row identifiers.
    """

    suspicious_columns = []

    for column in df.columns:

        column_data = df[column]

        # -------------------------
        # Numeric columns only
        # -------------------------

        if not np.issubdtype(
            column_data.dtype,
            np.number
        ):

            continue

        # -------------------------
        # Check uniqueness ratio
        # -------------------------

        unique_ratio = (
            column_data.nunique() / len(df)
        )

        highly_unique = unique_ratio > 0.95

        # -------------------------
        # Check sequential pattern
        # -------------------------

        sequential = np.all(
            np.diff(column_data) == 1
        )

        # -------------------------
        # Flag suspicious columns
        # -------------------------

        if highly_unique and sequential:

            suspicious_columns.append(column)

    # -------------------------
    # Print Results
    # -------------------------

    print("\nPotential Index-Like Columns:")
    print("-" * 40)

    if suspicious_columns:

        print(suspicious_columns)

    else:

        print("None detected.")

    return suspicious_columns

In [51]:
#-------------------------
#Testing Index-Like Columns
#-------------------------
suspicious_columns = detect_index_like_columns(df)


Potential Index-Like Columns:
----------------------------------------
['Index']


# Missing Value Detection

In [52]:
def analyze_missing_values(df):

    """
    Analyze missing values in a dataset.
    """

    # -------------------------
    # Missing Value Counts
    # -------------------------

    missing_counts = df.isnull().sum()

    # -------------------------
    # Missing Value Percentages
    # -------------------------

    missing_percentages = (
        missing_counts / len(df)
    ) * 100

    # -------------------------
    # Create Summary Table
    # -------------------------

    missing_summary = pd.DataFrame({

        "Missing Count": missing_counts,

        "Missing Percentage":
            missing_percentages

    })

    # -------------------------
    # Filter Columns With Missing Data
    # -------------------------

    missing_summary = missing_summary[
        missing_summary["Missing Count"] > 0
    ]

    # -------------------------
    # Sort Highest Missing First
    # -------------------------

    missing_summary = missing_summary.sort_values(

        by="Missing Percentage",

        ascending=False

    )

    # -------------------------
    # Print Results
    # -------------------------

    print("\nMissing Value Analysis")
    print("-" * 40)

    if len(missing_summary) == 0:

        print("No missing values detected.")

    else:

        print(missing_summary)

    return missing_summary

In [53]:
#-------------------------
# Testing Missing Values
#-------------------------
analyze_missing_values(df)


Missing Value Analysis
----------------------------------------
            Missing Count  Missing Percentage
Unnamed: 0             11               100.0


,Missing Count,Missing Percentage
Unnamed: 0,11,100.0


# Controlled Data Cleaning

In [54]:
def remove_empty_rows_and_columns(df):

    """
    Remove fully empty rows and columns.
    """

    original_shape = df.shape

    # -------------------------
    # Remove Fully Empty Rows
    # -------------------------

    df = df.dropna(
        axis=0,
        how="all"
    )

    # -------------------------
    # Remove Fully Empty Columns
    # -------------------------

    df = df.dropna(
        axis=1,
        how="all"
    )

    # -------------------------
    # Report Changes
    # -------------------------

    new_shape = df.shape

    removed_rows = (
        original_shape[0] - new_shape[0]
    )

    removed_columns = (
        original_shape[1] - new_shape[1]
    )

    print("\nEmpty Row/Column Cleanup")
    print("-" * 40)

    print(
        f"Removed Rows: {removed_rows}"
    )

    print(
        f"Removed Columns: {removed_columns}"
    )

    print(
        f"New Shape: {df.shape}"
    )

    return df

In [55]:
df = remove_empty_rows_and_columns(df)
df.head()


Empty Row/Column Cleanup
----------------------------------------
Removed Rows: 0
Removed Columns: 1
New Shape: (11, 7)


,Index,Material,Cake Volume (cubic inches),Surface Area cubic inches,Nitrogen Flow (L/min),Vacuum (mbar),Drying Time (minutes)
0,1,GDC-1234,75,7.5,60,100,240
1,2,Naproxen,100,10.0,70,120,260
2,3,Acetopminiphen,125,12.5,60,250,300
3,4,GDC-1234,300,30.0,50,240,600
4,5,GDC-9876,125,12.5,45,100,500


# User Defines Target Column

In [56]:
def select_target_column(df):

    """
    Allow user to select target column.
    """

    print("\nAvailable Columns")
    print("-" * 40)

    for index, column in enumerate(df.columns):

        print(f"{index}: {column}")

    while True:

        try:

            user_input = int(

                input(
                    "\nEnter target column index: "
                )

            )

            target_column = df.columns[user_input]

            print("\nSelected Target Column:")
            print(target_column)

            return target_column

        except:

            print(
                "\nInvalid selection."
            )

            print(
                "Please try again."
            )

In [57]:
target_column = select_target_column(df)


Available Columns
----------------------------------------
0: Index
1: Material
2: Cake Volume (cubic inches)
3: Surface Area cubic inches
4: Nitrogen Flow (L/min)
5: Vacuum (mbar)
6: Drying Time (minutes)



Enter target column index:  6



Selected Target Column:
Drying Time (minutes)


# Get Similarities of Column Names

In [63]:
# -------------------------
# Semantic Anchor Concepts
# -------------------------

REGRESSION_CONCEPTS = [

    "time",
    "duration",
    "temperature",
    "pressure",
    "yield",
    "mass",
    "volume",
    "size",
    "energy",
    "distance",
    "speed",
    "measurement",
    "continuous value",
    "scientific measurement",
    "process variable"

]

CLASSIFICATION_CONCEPTS = [

    "class",
    "category",
    "type",
    "group",
    "status",
    "label",
    "pass fail",
    "classification",
    "discrete category",
    "class label"

]

### Below is an example of what we are about to do using .encode and cosine_similarity

embedding_1 = model.encode(
    "Drying Time"
)

embedding_2 = model.encode(
    "Process Duration"
)

similarity = cosine_similarity(

    [embedding_1],

    [embedding_2]

)

print(similarity)

# Semantic Embeddings

In [64]:
# -------------------------
# Precompute Semantic Embeddings
# -------------------------

REGRESSION_EMBEDDINGS = {

    concept: model.encode(concept)

    for concept in REGRESSION_CONCEPTS

}

CLASSIFICATION_EMBEDDINGS = {

    concept: model.encode(concept)

    for concept in CLASSIFICATION_CONCEPTS

}

# Similarty Function

In [70]:
# -------------------------
# Semantic Similarity Scoring
# -------------------------

def semantic_similarity_score(

    target_text,

    embedding_dictionary

):

    """
    Compute semantic similarity between
    target text and anchor concepts.
    """

    target_embedding = model.encode(
        target_text
    )

    best_similarity = -1

    best_concept = None

    for concept, embedding in embedding_dictionary.items():

        similarity = cosine_similarity(

            [target_embedding],

            [embedding]

        )[0][0]

        if similarity > best_similarity:

            best_similarity = similarity

            best_concept = concept

    return {

        "best_similarity": float(best_similarity),

        "best_concept": best_concept

    }

In [72]:
#---------------------------
#Testing Snimilarity Scores
#---------------------------

# -------------------------
# Regression Semantic Match
# -------------------------

regression_result = semantic_similarity_score(

    "Drying Time",

    REGRESSION_EMBEDDINGS

)

print("\nRegression Semantic Match")
print("-" * 40)

print(
    f"Best Concept: "
    f"{regression_result['best_concept']}"
)

print(
    f"Similarity: "
    f"{regression_result['best_similarity']:.3f}"
)


# -------------------------
# Classification Semantic Match
# -------------------------

classification_result = semantic_similarity_score(

    "Drying Time",

    CLASSIFICATION_EMBEDDINGS

)

print("\n\nClassification Semantic Match")
print("-" * 40)

print(
    f"Best Concept: "
    f"{classification_result['best_concept']}"
)

print(
    f"Similarity: "
    f"{classification_result['best_similarity']:.3f}"
)


Regression Semantic Match
----------------------------------------
Best Concept: duration
Similarity: 0.318


Classification Semantic Match
----------------------------------------
Best Concept: status
Similarity: 0.149


# Define Problem as Classification or Regression

In [73]:
def infer_ml_task(df, target_column):

    """
    Infer machine learning task type
    using probabilistic heuristic voting.
    """

    target_data = df[target_column].dropna()

    regression_score = 0
    classification_score = 0

    evidence = []

    # -------------------------
    # Basic Statistics
    # -------------------------

    unique_values = target_data.nunique()

    total_values = len(target_data)

    unique_ratio = (
        unique_values / total_values
    )

    # -------------------------
    # Semantic Similarity Analysis
    # -------------------------

    regression_semantic = semantic_similarity_score(

        target_column,

        REGRESSION_EMBEDDINGS

    )

    classification_semantic = semantic_similarity_score(

        target_column,

        CLASSIFICATION_EMBEDDINGS

    )

    regression_similarity = (
        regression_semantic["best_similarity"]
    )

    classification_similarity = (
        classification_semantic["best_similarity"]
    )

    # Weighted semantic voting

    regression_score += (
        regression_similarity * 5
    )

    classification_score += (
        classification_similarity * 5
    )

    evidence.append(

        f"Regression semantic match: "

        f"{regression_semantic['best_concept']} "

        f"({regression_similarity:.3f})"

    )

    evidence.append(

        f"Classification semantic match: "

        f"{classification_semantic['best_concept']} "

        f"({classification_similarity:.3f})"

    )

    # -------------------------
    # Numeric vs Non-Numeric
    # -------------------------

    if np.issubdtype(
        target_data.dtype,
        np.number
    ):

        regression_score += 1

        evidence.append(
            "Target is numeric."
        )

        # -------------------------
        # Float vs Integer
        # -------------------------

        if np.issubdtype(
            target_data.dtype,
            np.floating
        ):

            regression_score += 2

            evidence.append(
                "Target contains floating point values."
            )

        elif np.issubdtype(
            target_data.dtype,
            np.integer
        ):

            classification_score += 1

            evidence.append(
                "Target contains integer values."
            )

        # -------------------------
        # Unique Ratio Analysis
        # -------------------------

        if unique_ratio > 0.5:

            regression_score += 2

            evidence.append(
                "Target has high unique value ratio."
            )

        elif unique_ratio < 0.1:

            classification_score += 2

            evidence.append(
                "Target has low unique value ratio."
            )

        else:

            regression_score += 1
            classification_score += 1

            evidence.append(
                "Target has moderate unique value ratio."
            )

        # -------------------------
        # Repeated Values
        # -------------------------

        repeated_ratio = (
            1 - unique_ratio
        )

        if repeated_ratio > 0.5:

            classification_score += 1

            evidence.append(
                "Target has many repeated values."
            )

        # -------------------------
        # Value Spacing Analysis
        # -------------------------

        sorted_values = np.sort(
            target_data.unique()
        )

        if len(sorted_values) > 2:

            differences = np.diff(
                sorted_values
            )

            evenly_spaced = np.all(

                differences == differences[0]

            )

            if evenly_spaced and unique_values <= 10:

                classification_score += 1

                evidence.append(
                    "Target has few evenly spaced values."
                )

            else:

                regression_score += 1

                evidence.append(
                    "Target values are not simple class-like spacing."
                )

    # -------------------------
    # Non-Numeric Targets
    # -------------------------

    else:

        classification_score += 4

        evidence.append(
            "Target is non-numeric."
        )

    # -------------------------
    # Probability Calculation
    # -------------------------

    total_score = (
        regression_score
        +
        classification_score
    )

    regression_probability = (
        regression_score / total_score
    )

    classification_probability = (
        classification_score / total_score
    )

    # -------------------------
    # Final Suggested Task
    # -------------------------

    if regression_probability >= classification_probability:

        suggested_task = "regression"

    else:

        suggested_task = "classification"

    # -------------------------
    # Print Results
    # -------------------------

    print("\nML Task Inference")
    print("-" * 40)

    print(
        f"Regression Score: "
        f"{regression_score:.3f}"
    )

    print(
        f"Classification Score: "
        f"{classification_score:.3f}"
    )

    print()

    print(
        f"Regression Probability: "
        f"{regression_probability:.3f}"
    )

    print(
        f"Classification Probability: "
        f"{classification_probability:.3f}"
    )

    print()

    print(
        f"Suggested Task: "
        f"{suggested_task}"
    )

    print("\nEvidence")
    print("-" * 40)

    for item in evidence:

        print(f"- {item}")

    # -------------------------
    # Return Results
    # -------------------------

    return {

        "suggested_task":
            suggested_task,

        "regression_probability":
            regression_probability,

        "classification_probability":
            classification_probability,

        "regression_score":
            regression_score,

        "classification_score":
            classification_score,

        "evidence":
            evidence

    }

In [74]:
task_type = infer_ml_task(
    df,
    target_column
)


ML Task Inference
----------------------------------------
Regression Score: 6.122
Classification Score: 1.441

Regression Probability: 0.809
Classification Probability: 0.191

Suggested Task: regression

Evidence
----------------------------------------
- Regression semantic match: duration (0.424)
- Classification semantic match: status (0.088)
- Target is numeric.
- Target contains integer values.
- Target has high unique value ratio.
- Target values are not simple class-like spacing.
